In [1]:
import pandas as pd
import numpy as np
import joblib
import folium
from folium.plugins import HeatMap

In [2]:
df = pd.read_csv("../data/processed/featured_uhi_v2.csv")
model = joblib.load("../outputs/lst_model_xgb_v3.pkl")

FEATURES = ["NDVI", "NDBI", "Elevation", "Population"]
df["LST_baseline"] = model.predict(df[FEATURES])

priority = pd.read_csv("../outputs/reports/intervention_priority.csv")
print(f"Loaded {len(df)} points and {len(priority)} priority zones")

Loaded 9893 points and 10 priority zones


In [3]:
center = [df["Latitude"].mean(), df["Longitude"].mean()]
m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")

heat_data = df[["Latitude", "Longitude", "LST_baseline"]].values.tolist()
HeatMap(
    heat_data,
    radius=12,
    blur=15,
    min_opacity=0.3
).add_to(m)

In [4]:
for rank, row in priority.iterrows():
    folium.Marker(
        location=[row["Latitude"], row["Longitude"]],
        popup=(f"Priority #{rank+1}<br>"
               f"Zone {row['zone']}<br>"
               f"Baseline LST: {row['LST_baseline']:.1f} C<br>"
               f"Strategy: {row['best_strategy']}<br>"
               f"Cooling: {row['best_cooling']:.2f} C"),
        icon=folium.Icon(color="darkred", icon="fire", prefix="fa")
    ).add_to(m)

m.save("../outputs/aurangabad_heat_map.html")
print("Saved -> outputs/aurangabad_heat_map.html")
m

Saved -> outputs/aurangabad_heat_map.html
